# preprocessing_prime -- run PRIME's own preprocessing on this project's data

Bridges `organize_neurone.m`'s BIDS output into `prime_core`'s **actual** preprocessing code,
and writes the result in the same format `preprocessing.m` produces, so that
`TEP_dipole_SpaTempFilt.ipynb` reads it with no changes beyond one path.

```
organize_neurone.m  ->  THIS NOTEBOOK  ->  TEP_dipole_SpaTempFilt.ipynb
 (per-block .set +      (prime_core's         (dipole filter +
  events.tsv)           Preprocessor)          single-trial N45)
```

**This notebook does not reimplement anything.** It imports `prime_core.preprocessing` and
calls `Preprocessor.add_trial` / `.calibrate()` / `.preprocess_post()` directly, so every
filtering, artifact, ICA, SOUND, SSP-SIR and trial-rejection decision is made by prime's own
code with prime's own `configs/prime.yaml`. If prime changes, re-running this picks the change
up automatically. All this notebook contributes is the **adapter**: turning per-block
continuous recordings plus `events.tsv` into the `(n_samples, n_channels)` buffers prime's API
expects, and turning prime's outputs back into `.set` + `.tsv`.

**Why it exists.** The deviation report found that `TEP_dipole_SpaTempFilt` was consuming
MATLAB `preprocessing.m` output while prime consumes its own Python chain -- different
baseline windows, artifact-blanking windows, SOUND/SSP-SIR parameters and trial-rejection
rules. Same dipole method, differently-cleaned data. This removes that difference entirely.

**What is genuinely different from `preprocessing.m`** (i.e. what you are switching to):

| | preprocessing.m (MATLAB) | this notebook (prime) |
|---|---|---|
| Post baseline | -20 to -10 ms | **-25 to -15 ms** |
| Artifact blanking | interpolate [-5,+6], [-5,+8] ms | **`fix_stim_artifact` [-14,+14], then to +15 ms** |
| SOUND | lambda 0.05, 5 iters | **lambda 0.01, 10 iters** |
| SSP-SIR | artScale automatic, PC 90th pct | **automatic/50, PC threshold 0.99** |
| Trial rejection | 3-stage `find_bad_trials` | **MAD z-score, global + local** |
| Ocular | ICLabel on pre-stim ICA copy | **ICA component z-threshold per trial** |
| Post rate | 5000 Hz (native) | **1000 Hz (polyphase)** |
| Post extent | -20 to 150 ms | **1 to 100 ms** |

**Sections**

1. Paths, prime import, and parameters
2. Load `organize_neurone` blocks and build one trial table
3. Forward solution (prime's recipe, built from this montage)
4. Epoch every block into prime's raw trial buffers
5. Calibrate prime's `Preprocessor` on the calibration block
6. Run `preprocess_post` on every trial, dropping rejects
7. Save `.set` + `.tsv` in `preprocessing.m`'s format
8. Diagnostics

## 1. Paths, prime import, and parameters

`prime_root` must point at the folder containing `prime_core` (i.e. `.../prime/decider`), so
that `import prime_core` resolves. Nothing is copied or vendored -- this is the live code.

`prime_core.prime_config` reads `configs/prime.yaml` through **omegaconf**, so that package
must be importable. If the import fails, `pip install omegaconf` is the fix; the cell below
says so explicitly rather than failing with a bare `ModuleNotFoundError`.

Dependencies. Install these into whichever environment runs this kernel:

```
pip install omegaconf scikit-learn eeglabio
```

- `omegaconf` -- prime's own `configs/prime.yaml` loader.
- `scikit-learn` -- prime's `DipoleFitter` imports `sklearn.metrics.r2_score`.
  **Install `scikit-learn`, not `sklearn`.** `pip install sklearn` fails with
  *"Failed to build 'sklearn' when getting requirements to build wheel"* -- that name is a
  deprecated placeholder package that now intentionally refuses to install, and the error is
  not a problem with your setup.
- `mne_icalabel` -- imported by prime's `utils/ica_calibrator.py`.
- `eeglabio` -- MNE needs it to *write* `.set` files (Section 7).

**NumPy 2.0.** prime calls `np.trapz`, which NumPy 2.0 removed (renamed `np.trapezoid`), so on
NumPy >= 2 prime's `calibrate()` fails with `module 'numpy' has no attribute 'trapz'`. The next
cell restores the alias before prime is used -- the two functions are numerically identical,
and this keeps prime itself untouched. No downgrade needed.

In [ ]:
import sys
from pathlib import Path

import numpy as np

# ---- NumPy 2.0 compatibility shim for prime -------------------------------------------
# prime's preprocessor.py:162 calls np.trapz() in its post-stimulus bad-channel (AUC)
# criterion. np.trapz was REMOVED in NumPy 2.0 and renamed np.trapezoid, so on NumPy >= 2
# prime's calibrate() dies with "module 'numpy' has no attribute 'trapz'".
#
# Restoring the alias here rather than editing prime keeps prime untouched (and keeps this
# notebook's promise that prime's code is the single source of truth). The two functions are
# numerically identical -- verified equal on the single-array, unit-spacing call prime makes.
# np.trapz is the ONLY NumPy-2 removal prime uses; the rest of its API is clean.
if not hasattr(np, 'trapz') and hasattr(np, 'trapezoid'):
    np.trapz = np.trapezoid
    print(f"NumPy {np.__version__}: restored np.trapz -> np.trapezoid alias for prime.")

# ---- where things live ---------------------------------------------------------------
bids_root  = Path(r"D:\Linus\Loop\BIDS")
prime_root = Path(r"D:\Linus\Loop\prime\decider")      # the folder CONTAINING prime_core
subject = 'Pilot002'
session = 'prime'

# Output: same layout preprocessing.m uses, but under a 'preprocessing_prime' derivative so
# the MATLAB results stay untouched and the two can be compared side by side.
out_dir = bids_root / 'derivatives' / 'preprocessing_prime' / f'sub-{subject}' / f'ses-{session}' / 'eeg'
out_dir.mkdir(parents=True, exist_ok=True)

# Cached forward solution (built once in Section 3, prime's own recipe).
fwd_path = bids_root / 'derivatives' / 'preprocessing_prime' / 'fsaverage-fwd.fif'
fwd_path.parent.mkdir(parents=True, exist_ok=True)

# ---- make prime importable -----------------------------------------------------------
if str(prime_root) not in sys.path:
    sys.path.insert(0, str(prime_root))

try:
    from prime_core.prime_config import (
        get_calibration_time_range, get_raw_sfreq, get_processed_sfreq,
        get_dipole_time_range, get_post_time_range, load_prime_config,
    )
except ModuleNotFoundError as e:
    if 'omegaconf' in str(e):
        raise ModuleNotFoundError(
            "prime_core needs 'omegaconf' to read configs/prime.yaml.\n"
            "Install it into this kernel's environment:  pip install omegaconf"
        ) from e
    raise ModuleNotFoundError(
        f"Could not import prime_core from {prime_root!s}.\n"
        "Check that prime_root points at the folder CONTAINING prime_core "
        "(i.e. .../prime/decider), not at prime_core itself."
    ) from e

from prime_core.preprocessing.preprocessor import Preprocessor, crop_mne_trial_to_buffer
from prime_core.preprocessing.build_fsaverage import COMMON_CHANNELS

# ---- everything below comes FROM prime, not from this notebook -----------------------
TRIAL_TMIN, TRIAL_TMAX = get_calibration_time_range()   # raw window fed to add_trial
RAW_SFREQ = get_raw_sfreq()
PROCESSED_SFREQ = get_processed_sfreq()
DIPOLE_TMIN, DIPOLE_TMAX = get_dipole_time_range()
POST_TMIN, POST_TMAX = get_post_time_range()

# ---- adapter-only settings (the only knobs this notebook itself owns) ----------------
# Blocks to process, in recording order. Matches preprocessing.m's task_list.
#
# 'evaluation-t60' is the 60-minute follow-up. It only exists for subjects whose session ran
# that far (Pilot002 onwards -- Pilot001 stopped at t30), so Section 2 SKIPS any task whose
# files are absent, with a loud warning, instead of raising. That keeps one TASKS list valid
# across subjects; drop a name from this list to exclude a block you do have.
TASKS = ['baseline', 'intervention-all',
         'evaluation-t0', 'evaluation-t15', 'evaluation-t30', 'evaluation-t60']
RUN = '01'
TMS_TRIGGER = 'A - Stimulation'     # the real TMS pulse (see organize_neurone.m's header)

# Conditions to drop before any processing.
#
# 'prime_triplet' trials are 50 Hz TRIPLETS: three TMS pulses at 0, ~20 and ~40 ms. Only the
# FIRST is written to events.tsv, so the 2nd and 3rd pulses are invisible in the trial table
# but very much present in the EEG -- measured on Pilot001, their artifacts reach ~640 uV and
# ~600 uV GFP, against ~1.5 uV for every clean condition. The third pulse lands at ~41 ms,
# i.e. INSIDE the N45 dipole window [38, 50] ms, so these trials cannot yield a valid N45 no
# matter how they are cleaned.
#
# Dropping them here also stops them contaminating a 'per_block' calibration (an intervention
# block calibrated on triplets would fit its ICA/SOUND/SSP-SIR to stimulator artifact), and
# saves ~40% of the processing time. Set to () to keep everything.
#
# Only the intervention blocks have a 'condition' column at all; baseline and every
# evaluation block carry no condition, so this filter never touches them.
EXCLUDE_CONDITIONS = ('prime_triplet',)

# Which trials calibrate the Preprocessor. prime calibrates on its calibration block.
CALIBRATION_STAGE = 'calibration'
N_CALIBRATION_TRIALS = None    # None = every calibration trial (prime offline attempts 125)

# How calibration state is distributed across blocks. See Section 5's markdown.
#   'prime_single' -- ONE calibration on the calibration block, applied to every trial in
#                     every block. This is what prime does and is the default. Note the
#                     asymmetry it creates in THIS dataset: baseline is recorded before the
#                     calibration block exists, and evaluation-t60 -- the block furthest from
#                     it -- starts a couple of hours after it.
#   'per_block'    -- a fresh Preprocessor calibrated on each block's own trials, then
#                     applied to that block only. Removes the drift/extrapolation concern
#                     and is closer in spirit to preprocessing.m, but is NO LONGER prime's
#                     method, and costs one full ICA+SOUND calibration per block (with
#                     evaluation-t60 present that is 10 blocks, so ~10x slower).
CALIBRATION_MODE = 'prime_single'
MIN_CALIBRATION_TRIALS = 30    # refuse to calibrate a block on fewer than this many trials

# prime's preprocess_post returns ONLY the dipole window [38,50] ms, because that is all the
# online system needs. For an offline notebook that is awkward: TEP_dipole_SpaTempFilt's
# diagnostics want surrounding context. 'post' widens the FINAL CROP ONLY to prime's own
# post window [1,100] ms -- every processing step before it is untouched and identical.
# 'dipole' reproduces prime byte-for-byte (13 samples). See Section 6.
SAVE_EXTENT = 'post'          # 'post' (recommended) or 'dipole'

print(f"prime_core imported from: {prime_root}")
print(f"  raw {RAW_SFREQ:.0f} Hz -> processed {PROCESSED_SFREQ:.0f} Hz")
print(f"  raw trial window   : [{TRIAL_TMIN*1000:.0f}, {TRIAL_TMAX*1000:.0f}] ms")
print(f"  post window        : [{POST_TMIN*1000:.0f}, {POST_TMAX*1000:.0f}] ms")
print(f"  dipole window      : [{DIPOLE_TMIN*1000:.0f}, {DIPOLE_TMAX*1000:.0f}] ms")
print(f"  saving extent      : {SAVE_EXTENT}")
print(f"  calibration mode   : {CALIBRATION_MODE}"
      + ("   (prime's own behaviour)" if CALIBRATION_MODE == 'prime_single'
         else "   (NOT prime's behaviour -- see Section 5)"))
print(f"  tasks requested    : {', '.join(TASKS)}")
print(f"Writing to: {out_dir}")

## 2. Load `organize_neurone` blocks and build one trial table

`organize_neurone.m` writes one **continuous** `.set` per block plus an `events.tsv` whose rows
are the real NeurOne triggers, with the matching `trials_*.csv` columns already attached. This
section reads each block, keeps only the `A - Stimulation` events, and concatenates their event
tables into one long provenance table in recording order -- the same ordering
`preprocessing.m` produces, and the ordering `TEP_dipole_SpaTempFilt` assumes.

The `block_label` column is what `TEP_dipole_SpaTempFilt`'s group queries select on, so it is
constructed here to match exactly what the MATLAB pipeline emitted: `baseline`, `calibration`,
`intervention_block_1..4`, `evaluation-t0/t15/t30/t60`. Note that the `intervention-all`
recording contains **both** the calibration trials and the four intervention blocks,
distinguished by its own `stage` column -- so one file becomes five `block_label`s, and the
ten blocks come from six recordings.

**Blocks a subject does not have are skipped, not fatal.** `evaluation-t60` exists from
Pilot002 onwards but not for Pilot001, so a task whose `.set`/`events.tsv` are absent is
reported and dropped from `TASKS` instead of raising, and the same Section 1 list runs for
every subject.

**The blocks do not all carry the same columns.** Only `intervention-all` has a `condition`
column (plus the PRIME performance columns -- `tep_amplitude`, `prediction_probability` and
so on); the evaluation blocks alone have `run_index`; `baseline` has neither. `pd.concat`
unions the columns and fills the gaps with `NaN`, *after* the per-file `.fillna('')` has
already run, so this cell re-normalises `condition` to `''`. Without that, both the
`EXCLUDE_CONDITIONS` filter and Section 8's `groupby('condition')` would quietly skip every
baseline and evaluation trial -- five of the ten blocks.

In [ ]:
import mne
import numpy as np
import pandas as pd

mne.set_log_level('ERROR')

eeg_dir = bids_root / f'sub-{subject}' / f'ses-{session}' / 'eeg'

# Blocks a subject genuinely does not have (e.g. evaluation-t60 for Pilot001) are skipped
# rather than fatal, so one TASKS list works across subjects. Everything downstream iterates
# TASKS, so narrowing it here is enough -- no other cell needs to know.
missing_tasks = []
present_tasks = []
for task in TASKS:
    stem = f'sub-{subject}_ses-{session}_task-{task}_run-{RUN}'
    if (eeg_dir / f'{stem}_eeg.set').exists() and (eeg_dir / f'{stem}_events.tsv').exists():
        present_tasks.append(task)
    else:
        missing_tasks.append(task)

if missing_tasks:
    print(f"!! NOT FOUND for sub-{subject}, skipping: {', '.join(missing_tasks)}")
    print(f"   (looked in {eeg_dir})")
    print("   If you expected these, run organize_neurone.m for this subject first.\n")
if not present_tasks:
    raise FileNotFoundError(
        f"None of TASKS were found in {eeg_dir}. Run organize_neurone.m for this subject.")

TASKS = present_tasks

raws, tables = {}, []
for task in TASKS:
    stem = f'sub-{subject}_ses-{session}_task-{task}_run-{RUN}'
    set_path = eeg_dir / f'{stem}_eeg.set'
    tsv_path = eeg_dir / f'{stem}_events.tsv'

    # preload=False: this section only needs the header (sfreq, duration, channel names).
    # The 52-minute intervention-all block is ~3.8 GB preloaded -- see Section 4.
    raws[task] = mne.io.read_raw_eeglab(str(set_path), preload=False)

    ev = pd.read_csv(tsv_path, sep='\t').fillna('')
    ev = ev[ev['trigger_type_neurone'] == TMS_TRIGGER].reset_index(drop=True)
    ev['task'] = task
    tables.append(ev)

    print(f"{task:<20} {raws[task].n_times/raws[task].info['sfreq']:7.1f} s, "
          f"{raws[task].info['sfreq']:.0f} Hz, {len(ev):4d} TMS trigger(s)")

trial_info = pd.concat(tables, ignore_index=True)

# The blocks carry DIFFERENT trials_*.csv column sets: only intervention-all has 'condition'
# (and the PRIME performance columns), while the evaluation blocks alone have 'run_index'.
# pd.concat unions the columns and fills the gaps with NaN -- note this happens AFTER the
# per-file .fillna('') above, so those NaNs survive. Left alone they would make
# trial_info['condition'].isin(...) and Section 8's groupby('condition') silently ignore
# every baseline and evaluation trial. Normalising to '' here keeps them visible.
if 'condition' not in trial_info.columns:
    trial_info['condition'] = ''
trial_info['condition'] = trial_info['condition'].fillna('').astype(str)


def _block_label(row):
    """Reproduce preprocessing.m's block_label vocabulary exactly."""
    if row['task'] != 'intervention-all':
        return row['task']
    return row['stage']          # 'calibration' or 'intervention_block_1..4'


trial_info['block_label'] = trial_info.apply(_block_label, axis=1)

# Sanity: sampling rate must be prime's raw_sfreq, or every downstream time index is wrong.
for task, raw in raws.items():
    if abs(raw.info['sfreq'] - RAW_SFREQ) > 1e-6:
        raise ValueError(
            f"{task} is at {raw.info['sfreq']:.0f} Hz but prime.yaml declares raw_sfreq="
            f"{RAW_SFREQ:.0f} Hz. organize_neurone.m must not resample.")

# --- drop conditions that cannot yield a valid TEP (see EXCLUDE_CONDITIONS) -------------
if EXCLUDE_CONDITIONS:
    _drop = trial_info['condition'].isin(EXCLUDE_CONDITIONS)
    if _drop.any():
        print(f"\nDropping {int(_drop.sum())} trial(s) in {list(EXCLUDE_CONDITIONS)}:")
        for c, n in trial_info.loc[_drop, 'condition'].value_counts().items():
            print(f"  {c:<24} {n:5d} trials")
        trial_info = trial_info[~_drop].reset_index(drop=True)

# prime's raw trial window is [-1.1, +0.1] s, so consecutive LOGGED pulses must be >1.2 s
# apart, or one trial's pre-stimulus window (which prime's ICA and QC calibrate on) would
# contain the previous pulse's artifact.
#
# NOTE: this checks the gaps between logged triggers only, and that is NOT sufficient on its
# own -- triplet trials fire three pulses but log one, so their 2nd/3rd pulses are invisible
# here. That is what EXCLUDE_CONDITIONS above handles, and what Section 8's post-hoc artifact
# scan double-checks against the actual cleaned data.
_need = TRIAL_TMAX - TRIAL_TMIN
for task in TASKS:
    onsets = np.sort(trial_info.loc[trial_info['task'] == task, 'onset'].to_numpy(float))
    if len(onsets) > 1:
        gap = np.diff(onsets).min()
        if gap < _need:
            raise ValueError(
                f"{task}: minimum inter-pulse gap is {gap:.3f} s but prime's trial window "
                f"spans {_need:.3f} s -- epochs would overlap and contaminate the "
                "pre-stimulus period.")

# Canonical block order for reporting. Any label NOT in this list is appended rather than
# dropped, so an unexpected stage value can never hide from the count below.
EXPECTED_BLOCK_ORDER = [
    'baseline', 'calibration',
    'intervention_block_1', 'intervention_block_2',
    'intervention_block_3', 'intervention_block_4',
    'evaluation-t0', 'evaluation-t15', 'evaluation-t30', 'evaluation-t60',
]
_seen = list(dict.fromkeys(trial_info['block_label']))
_report_order = ([b for b in EXPECTED_BLOCK_ORDER if b in _seen]
                 + [b for b in _seen if b not in EXPECTED_BLOCK_ORDER])

print(f"\nTotal TMS trials across all blocks: {len(trial_info)}")
print(trial_info['block_label'].value_counts().reindex(_report_order).to_string())

## 3. Forward solution (prime's recipe)

`prime_core.preprocessing.build_fsaverage` builds prime's forward solution from a specific
subject file that lives in prime's own `offline_data/`, which this project does not have. The
cell below runs **the same recipe** -- fsaverage, `fsaverage-ico-5-src.fif`,
`fsaverage-5120-5120-5120-bem-sol.fif`, `standard_1005`, `COMMON_CHANNELS` (imported from
prime, not retyped) -- against this montage instead, and caches the result.

Everything downstream that touches the head model (`Preprocessor`'s channel ordering, and
`DipoleFitter`'s leadfield in `TEP_dipole_SpaTempFilt`) then uses this one file, so the
channel order is fixed by the forward and never inferred.

In [ ]:
import os

if fwd_path.exists():
    forward = mne.read_forward_solution(str(fwd_path), verbose=False)
    print(f"Using cached forward solution: {fwd_path}")
else:
    fs_dir = mne.datasets.fetch_fsaverage(verbose=True)
    bem_dir = Path(fs_dir) / 'bem'

    # Same three assets build_fsaverage.py uses.
    trans = 'fsaverage'
    src = mne.read_source_spaces(str(bem_dir / 'fsaverage-ico-5-src.fif'))
    bem = mne.read_bem_solution(str(bem_dir / 'fsaverage-5120-5120-5120-bem-sol.fif'))

    # An Info carrying exactly prime's channel set, in prime's own order.
    template = raws[TASKS[0]].copy().pick(COMMON_CHANNELS)
    template.reorder_channels(COMMON_CHANNELS)
    template.set_montage(mne.channels.make_standard_montage('standard_1005'))

    print("Computing forward solution (a few minutes)...")
    forward = mne.make_forward_solution(template.info, trans, src, bem, verbose=True)
    mne.write_forward_solution(str(fwd_path), forward, overwrite=True)
    print(f"Wrote {fwd_path}")

FWD_CH_NAMES = list(forward.ch_names)
print(f"\nForward: {len(FWD_CH_NAMES)} channels, "
      f"{forward['nsource']} sources, coord frame = head")

missing = [ch for ch in FWD_CH_NAMES if ch not in raws[TASKS[0]].ch_names]
if missing:
    raise ValueError(f"Recording is missing channels the forward needs: {missing}")
print(f"All {len(FWD_CH_NAMES)} forward channels present in the recording.")

## 4. Index every trial (read on demand, not all at once)

prime's online API takes one trial at a time as a plain `(n_samples, n_channels)` array at
5000 Hz, plus the time of each sample relative to the pulse. This section works out *where*
every trial lives -- which block, which sample offset -- but deliberately **does not read any
EEG yet**.

**Why it is built this way.** Pilot002 logs 1,425 TMS pulses across its six recordings (945
once `EXCLUDE_CONDITIONS` has removed the triplets). Materialising all 1,425 up front costs
`1425 x 60 ch x 6001 samples x 8 bytes = 4.1 GB` of buffers, and on the 52-minute
`intervention-all` block the preloaded raw (~3.8 GB) plus an `mne.Epochs(preload=True)`
(~2.7 GB) plus the accumulating buffers peaks near **10 GB** -- which on most machines means
swapping, and looks exactly like the notebook hanging. The processed output that is actually
needed is `945 x 60 x 100 x 8 =` **~45 MB**. So trials are read one at a time, straight out of
the on-disk `.fdt` via `raw[channels, start:stop]`, and thrown away immediately after prime
has consumed them. Peak memory becomes one trial (~2.9 MB) plus the outputs.

`PRELOAD_RAW = True` restores the old behaviour if you have the RAM and would rather trade it
for fewer disk reads; the default `False` is the safe choice.

Two details that matter and are easy to get wrong:

- **Channel order is taken from the forward solution**, not from the recording. prime's
  `Preprocessor` builds its internal `Info` from `forward.ch_names` and validates only the
  channel *count*, so a recording in a different order would be silently mis-assigned. The
  reader below indexes channels by explicit position in `FWD_CH_NAMES`, so the order is
  correct by construction rather than by convention. This also means each block is checked
  independently -- `evaluation-t60` having a different channel order than `baseline` would be
  caught here rather than corrupting its trials.
- **Sample arithmetic matches `epoch_n_times`.** The window is
  `round((tmax - tmin) * sfreq) + 1` samples starting at `round(tmin * sfreq)` from the pulse,
  which is exactly what MNE's `crop(..., include_tmax=True)` and prime's `crop_eeg_buffer`
  produce -- verified against `crop_mne_trial_to_buffer` on a sample trial at the end of the
  cell.

In [ ]:
from prime_core.prime_config import epoch_n_times, time_to_sample

PRELOAD_RAW = False        # see markdown -- False keeps peak memory at ~1 trial

montage = mne.channels.make_standard_montage('standard_1005')

# Re-open each block, this time keeping only the forward's channels. With preload=False these
# are cheap on-disk handles; slicing them reads just the requested window.
readers, ch_index = {}, {}
for task in TASKS:
    stem = f'sub-{subject}_ses-{session}_task-{task}_run-{RUN}'
    r = mne.io.read_raw_eeglab(str(eeg_dir / f'{stem}_eeg.set'), preload=PRELOAD_RAW)
    readers[task] = r
    # Positional index of each forward channel within THIS block's channel list.
    ch_index[task] = [r.ch_names.index(ch) for ch in FWD_CH_NAMES]

N_TRIAL_SAMPLES = epoch_n_times(TRIAL_TMIN, TRIAL_TMAX, RAW_SFREQ)
TRIAL_START_OFFSET = time_to_sample(TRIAL_TMIN, 0.0, RAW_SFREQ)      # negative
TRIAL_TIMESTAMPS = TRIAL_TMIN + np.arange(N_TRIAL_SAMPLES) / RAW_SFREQ

# Per-trial location: (task, first sample of the trial window). No EEG read here.
trial_task = trial_info['task'].to_numpy()
trial_start = np.empty(len(trial_info), dtype=np.int64)
for task in TASKS:
    m = trial_task == task
    onset_samp = np.round(trial_info.loc[m, 'onset'].to_numpy(float) * RAW_SFREQ).astype(np.int64)
    trial_start[m] = onset_samp + TRIAL_START_OFFSET

# Bounds check now, so a trial running off the end of a recording fails here rather than
# halfway through a multi-minute processing loop.
for task in TASKS:
    m = trial_task == task
    n_times = readers[task].n_times
    lo, hi = trial_start[m].min(), trial_start[m].max() + N_TRIAL_SAMPLES
    if lo < 0 or hi > n_times:
        raise ValueError(
            f"{task}: trial window [{lo}, {hi}) falls outside the recording "
            f"[0, {n_times}). Check the first/last onset in events.tsv.")


def trial_buffer(row):
    """Read one trial as (n_samples, n_channels) in forward-solution channel order."""
    task = trial_task[row]
    start = int(trial_start[row])
    data, _ = readers[task][ch_index[task], start:start + N_TRIAL_SAMPLES]
    return np.ascontiguousarray(data.T, dtype=np.float64), TRIAL_TIMESTAMPS


print(f"Indexed {len(trial_info)} trials, {N_TRIAL_SAMPLES} samples each "
      f"([{TRIAL_TMIN*1000:.0f}, {TRIAL_TMAX*1000:.0f}] ms at {RAW_SFREQ:.0f} Hz), "
      f"preload={PRELOAD_RAW}")
for task in TASKS:
    print(f"  {task:<20} {int((trial_task == task).sum()):4d} trials")

_buf, _ts = trial_buffer(0)
print(f"\nOne trial: {_buf.shape} float64 = {_buf.nbytes/1e6:.1f} MB "
      f"(vs {len(trial_info)*_buf.nbytes/1e9:.1f} GB if all were held at once)")

# The reader must agree exactly with prime's own cropping helper.
_ev = trial_info[trial_info['task'] == TASKS[0]]
_r = readers[TASKS[0]]
_events = np.column_stack([
    np.round(_ev['onset'].to_numpy(float)[:1] * RAW_SFREQ).astype(int) + _r.first_samp,
    [0], [1]])
_ep = mne.Epochs(_r, _events, tmin=TRIAL_TMIN, tmax=TRIAL_TMAX, baseline=None,
                 preload=True, reject=None, flat=None, reject_by_annotation=False,
                 picks=ch_index[TASKS[0]], verbose=False)
_ref_buf, _ref_ts = crop_mne_trial_to_buffer(_ep[0], TRIAL_TMIN, TRIAL_TMAX)
assert _ref_buf.shape == _buf.shape, f"{_ref_buf.shape} vs {_buf.shape}"
assert np.allclose(_ref_buf, _buf), "reader disagrees with crop_mne_trial_to_buffer"
assert np.allclose(_ref_ts, _ts), "timestamps disagree with crop_mne_trial_to_buffer"
print("Reader verified against prime's crop_mne_trial_to_buffer on trial 0.")
del _ep, _ref_buf, _buf

## 5. Calibrate prime's `Preprocessor`

`Preprocessor.calibrate()` is what fixes every parameter used to clean the rest of the
recording: the bad-channel list and its interpolation matrix, the ICA solution and which of
its components count as ocular, the SOUND filter, the SSP-SIR projectors, and the MAD
rejection statistics each later trial is judged against. All of it is prime's code.

### `CALIBRATION_MODE` -- and why this dataset makes the choice matter

prime is **stateful by design**: calibrate once at the start of a session, then apply that
frozen state to every trial that follows. That fits prime's own use case, where calibration is
immediately followed by the intervention it was calibrated for. This dataset is laid out
differently, and two consequences are worth being deliberate about:

- **`baseline` is recorded *before* the calibration block exists.** Applying calibration state
  to it means extrapolating an ICA/SOUND/SSP-SIR solution *backwards* in time.
- **The evaluation blocks are far away, and `evaluation-t60` is the extreme case.**
  Calibration ends ~8.6 min into `intervention-all`. `evaluation-t30` starts well over an hour
  after that, and `evaluation-t60` is a further ~30 min beyond it -- roughly two hours of
  frozen calibration state by the time the last block is cleaned. Over that span, impedance
  drift and cap movement mean the bad-channel list and ICA components may no longer describe
  the data. Adding t60 does not create this problem, but it does extend the lever arm, so it
  is the block to watch in the rejection table Section 6 prints.

The two modes:

| | `'prime_single'` (default) | `'per_block'` |
|---|---|---|
| Calibrations | 1, on the calibration block | 1 per block, on that block's own trials |
| Applied to | every trial in every block | that block only |
| Matches prime | **yes** | no |
| Drift exposure | grows with time since calibration | none |
| Cost | one ICA + SOUND | ~10x that (10 blocks with t60) |

**`'prime_single'` is the default and is what you want if the goal is comparability with the
online PRIME pipeline** -- which is the whole reason this notebook exists. `'per_block'` is
provided so the drift question can be answered empirically rather than argued about: run both,
compare the resulting N45 amplitudes, and if `baseline` and the late evaluation blocks
(`evaluation-t30`, `evaluation-t60`) move appreciably while the intervention blocks do not,
drift was real. Note that `'per_block'` is closer in spirit to `preprocessing.m` (which
derives its parameters from the pooled recording) but is no longer prime's method, so results
from it are not prime-comparable.

In [ ]:
import time

BLOCK_ORDER = list(dict.fromkeys(trial_info['block_label']))    # recording order, deduped


def _calibrate_on(rows, label):
    """Build a fresh prime Preprocessor and calibrate it on the given trial rows."""
    if len(rows) < MIN_CALIBRATION_TRIALS:
        raise RuntimeError(
            f"Block {label!r} has only {len(rows)} trial(s) to calibrate on "
            f"(MIN_CALIBRATION_TRIALS = {MIN_CALIBRATION_TRIALS}). prime's ICA/SOUND need a "
            "reasonable number of trials; use CALIBRATION_MODE = 'prime_single', or lower "
            "the threshold if you know what you are doing.")

    pp = Preprocessor(str(fwd_path))
    for k in rows:
        pp.add_trial(*trial_buffer(k))

    t0 = time.perf_counter()
    mb, db = pp.calibrate()
    prm = pp.calibration_params
    print(f"  {label:<22} {len(rows):4d} in -> {mb.shape[0]:4d} survived   "
          f"bad ch: {len(prm['bad_channels'])}, ocular IC: "
          f"{len(prm['ocular_threshold_post'])}   ({time.perf_counter()-t0:.0f} s)")
    return pp, mb, db


preprocessors, cal_results = {}, {}

if CALIBRATION_MODE == 'prime_single':
    rows = np.flatnonzero((trial_info['block_label'] == CALIBRATION_STAGE).to_numpy())
    if len(rows) == 0:
        raise RuntimeError(f"No trials with block_label == {CALIBRATION_STAGE!r}.")
    if N_CALIBRATION_TRIALS is not None:
        rows = rows[:N_CALIBRATION_TRIALS]
    print(f"ONE calibration on {CALIBRATION_STAGE!r} (rows {rows[0]}..{rows[-1]}), "
          f"applied to all {len(BLOCK_ORDER)} blocks:")
    pp, mb, db = _calibrate_on(rows, CALIBRATION_STAGE)
    preprocessors['__shared__'] = pp
    cal_results[CALIBRATION_STAGE] = dict(model=mb, dipole=db, rows=rows)

elif CALIBRATION_MODE == 'per_block':
    print(f"SEPARATE calibration per block ({len(BLOCK_ORDER)} total) -- NOT prime's behaviour:")
    for label in BLOCK_ORDER:
        rows = np.flatnonzero((trial_info['block_label'] == label).to_numpy())
        if N_CALIBRATION_TRIALS is not None:
            rows = rows[:N_CALIBRATION_TRIALS]
        pp, mb, db = _calibrate_on(rows, label)
        preprocessors[label] = pp
        cal_results[label] = dict(model=mb, dipole=db, rows=rows)

else:
    raise ValueError(f"Unknown CALIBRATION_MODE: {CALIBRATION_MODE!r}")


def _pp_for(block_label):
    """The Preprocessor whose calibration state governs this block."""
    return preprocessors.get(block_label) or preprocessors['__shared__']


print(f"\n{len(preprocessors)} calibrated Preprocessor(s) ready.")

## 6. Run `preprocess_post` on every trial

Every trial in the recording -- all blocks, including the calibration trials themselves -- now
goes through `preprocess_post()` with a frozen calibration state. Which state depends on
`CALIBRATION_MODE`: under `'prime_single'` there is one shared `Preprocessor` for everything,
under `'per_block'` each trial is routed to the one calibrated on its own block (`_pp_for`).
prime returns `None` for a trial it rejects (ocular ICA z-threshold, or the global/local MAD
checks), and those are dropped here with their indices logged, exactly as `preprocessing.m`
drops its own rejects. The surviving trials stay row-for-row aligned with the filtered trial
table.

The **rejections-by-block table** printed at the end is the main diagnostic for the drift
question raised in Section 5. Under `'prime_single'`, a rejection rate that climbs steadily
from `baseline` through the intervention blocks to `evaluation-t30` and `evaluation-t60` is
the signature of calibration state going stale -- and t60, ~2 h past calibration, is where
that would show up first and most clearly. A flat rate across all ten blocks means the single
calibration is holding up fine and the default mode is safe here.

**About `SAVE_EXTENT`.** prime's `preprocess_post` ends with
`epoch_post.crop(self._dipole_tmin, self._dipole_tmax)`, returning just the 13-sample dipole
window -- all the online system consumes. With `SAVE_EXTENT = 'post'` this notebook overrides
those two attributes on the instance so the **final crop only** widens to prime's own post
window `[1, 100] ms`. Nothing upstream changes: `calibrate()` reads the dipole range from
`prime_config` directly rather than from these attributes, so the calibration above is
unaffected, and every filtering/artifact/rejection step still runs identically. The wider
output simply keeps enough context for `TEP_dipole_SpaTempFilt`'s diagnostic plots to be
readable. Set `SAVE_EXTENT = 'dipole'` to get prime's literal 13-sample output instead.

In [ ]:
if SAVE_EXTENT == 'post':
    # Widen ONLY preprocess_post's final crop (see markdown above), on EVERY preprocessor.
    for _pp in preprocessors.values():
        _pp._dipole_tmin = POST_TMIN
        _pp._dipole_tmax = POST_TMAX
    save_tmin, save_tmax = POST_TMIN, POST_TMAX
elif SAVE_EXTENT == 'dipole':
    save_tmin, save_tmax = DIPOLE_TMIN, DIPOLE_TMAX
else:
    raise ValueError(f"Unknown SAVE_EXTENT: {SAVE_EXTENT!r}")

print(f"Saving epochs over [{save_tmin*1000:.0f}, {save_tmax*1000:.0f}] ms "
      f"at {PROCESSED_SFREQ:.0f} Hz, mode = {CALIBRATION_MODE}.\n")

block_of = trial_info['block_label'].to_numpy()
n_trials_total = len(trial_info)
post_list, kept_rows, rejected_rows = [], [], []
t0 = time.perf_counter()
for k in range(n_trials_total):
    # Trials are read from disk one at a time and released immediately (Section 4).
    out = _pp_for(block_of[k]).preprocess_post(*trial_buffer(k))
    if out is None:
        rejected_rows.append(k)
    else:
        post_list.append(out)
        kept_rows.append(k)
    if (k + 1) % 100 == 0:
        _el = time.perf_counter() - t0
        _eta = _el / (k + 1) * (n_trials_total - k - 1)
        print(f"  {k+1:4d}/{n_trials_total} trials  "
              f"({len(rejected_rows)} rejected so far, "
              f"{_el:.0f} s elapsed, ~{_eta:.0f} s left)")

if not post_list:
    raise RuntimeError("Every trial was rejected -- check the calibration block.")

post_data = np.stack(post_list, axis=0)          # (n_kept, n_channels, n_times)
kept_rows = np.asarray(kept_rows)
rejected_rows = np.asarray(rejected_rows, dtype=int)

trial_info_kept = trial_info.iloc[kept_rows].reset_index(drop=True)
trial_info_kept.insert(0, 'orig_trial_index', kept_rows)
# Provenance: which calibration state actually cleaned each surviving trial.
trial_info_kept['calib_mode'] = CALIBRATION_MODE
trial_info_kept['calib_source'] = (
    CALIBRATION_STAGE if CALIBRATION_MODE == 'prime_single'
    else trial_info_kept['block_label'])

print(f"\nprocess_post finished in {time.perf_counter() - t0:.0f} s")
print(f"  kept     : {len(kept_rows)} / {n_trials_total} trials")
print(f"  rejected : {len(rejected_rows)} ({100*len(rejected_rows)/n_trials_total:.1f}%)")
print(f"  data     : {post_data.shape}")

# Where did the rejections land? A block-by-block breakdown is far more informative than a
# single overall percentage -- one block rejecting far more than the others is a real signal.
rej_by_block = (trial_info.assign(rejected=~trial_info.index.isin(kept_rows))
                .groupby('block_label')['rejected'].agg(['sum', 'count']))
rej_by_block['pct'] = 100 * rej_by_block['sum'] / rej_by_block['count']
print("\nRejections by block:")
print(rej_by_block.rename(columns={'sum': 'rejected', 'count': 'n_trials'}).to_string())

## 7. Save `.set` + `.tsv`

Written with the same filenames and the same two-file convention `preprocessing.m` uses, so
`TEP_dipole_SpaTempFilt.ipynb` runs against this output by changing only its `preproc_dir` in
Section 1 -- pointing it at `derivatives/preprocessing_prime/` instead of
`derivatives/preprocessing/`.

The epochs are already at 1000 Hz and already average-referenced by prime, so
`TEP_dipole_SpaTempFilt`'s own resample and crop steps become no-ops (its crop bounds get
clipped to what is present, which it reports at runtime). That is the intended outcome: with
this input, every preprocessing decision has been made by prime and the dipole notebook does
nothing but fit.

A third file, `*_rejected_trials.tsv`, records exactly which original trial indices prime
discarded and what block they came from -- worth keeping, since prime's rejection is per-trial
and stateful, and is not reproducible from the saved epochs alone.

In [ ]:
info_out = mne.create_info(ch_names=FWD_CH_NAMES, sfreq=PROCESSED_SFREQ, ch_types='eeg')
info_out.set_montage(montage)

epochs_out = mne.EpochsArray(
    post_data, info_out,
    events=np.column_stack([np.arange(len(post_data)),
                            np.zeros(len(post_data), int),
                            np.ones(len(post_data), int)]),
    tmin=save_tmin, verbose=False,
)

set_name = f'sub-{subject}_ses-{session}_desc-poststim_eeg.set'
tsv_name = f'sub-{subject}_ses-{session}_desc-preproc_epochs.tsv'
rej_name = f'sub-{subject}_ses-{session}_desc-rejected_trials.tsv'

epochs_out.export(str(out_dir / set_name), fmt='eeglab', overwrite=True)
trial_info_kept.to_csv(out_dir / tsv_name, sep='\t', index=False)

pd.DataFrame({
    'orig_trial_index': rejected_rows,
    'block_label': trial_info['block_label'].to_numpy()[rejected_rows] if len(rejected_rows) else [],
    'task': trial_info['task'].to_numpy()[rejected_rows] if len(rejected_rows) else [],
}).to_csv(out_dir / rej_name, sep='\t', index=False)

print("Saved:")
for f in (set_name, tsv_name, rej_name):
    print(f"  {out_dir / f}")

print(f"\nTo use this in TEP_dipole_SpaTempFilt.ipynb, set in its Section 1:")
print(f"    preproc_dir = bids_root / 'derivatives' / 'preprocessing_prime' / "
      f"f'sub-{{subject}}' / f'ses-{{session}}' / 'eeg'")

# The invariant that notebook asserts on load -- check it here so a mismatch surfaces now.
assert len(epochs_out) == len(trial_info_kept), (
    f"{len(epochs_out)} epochs vs {len(trial_info_kept)} trial_info rows")
print(f"\nAlignment check passed: {len(epochs_out)} epochs == {len(trial_info_kept)} rows.")

## 8. Diagnostics

Four checks worth doing before trusting the output downstream:

1. **Butterfly of the calibration average** with the dipole window shaded -- is there a real
   TEP where `TEP_dipole_SpaTempFilt` is about to look?
2. **Per-block averages** -- do all ten blocks look broadly comparable, or did one come out
   obviously different (which usually means many of its trials were rejected, or the
   calibration state does not suit it)? `evaluation-t60` is the one furthest from calibration,
   so it is the block most likely to separate from the rest under `'prime_single'`.
3. **Residual-artifact scan.** Every surviving group's average GFP is checked inside the
   dipole window against the quietest group. A stimulator artifact is orders of magnitude
   larger than a TEP, so anything more than ~10x the quiet baseline is almost certainly a
   pulse, not brain activity -- this is what catches trials whose *unlogged* extra pulses land
   in the analysis window. On Pilot001 the `prime_triplet` condition failed this badly
   (~640 uV vs ~1.5 uV) before `EXCLUDE_CONDITIONS` removed it. Grouping is by `condition`
   where one exists and by `block_label` otherwise, so `baseline` and the four
   `evaluation-*` blocks -- which have no `condition` column at all -- are each scanned
   separately rather than pooled into one uninformative row.
4. **Cross-check against prime's own `dipole_buffers`** from Section 5. The calibration trials
   are processed twice by construction -- once inside `calibrate()`, once again through
   `preprocess_post` -- so the dipole-window data should agree closely. It will not agree
   *exactly*: `calibrate()` computes SOUND/SSP-SIR on the trial ensemble, while
   `preprocess_post` applies the resulting fixed operators per trial. A small residual is
   expected and healthy; a large one means the calibration state is not transferring.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(11, 7))

cal_kept = trial_info_kept['block_label'] == CALIBRATION_STAGE
cal_avg = post_data[cal_kept.to_numpy()].mean(axis=0)

axes[0].plot(epochs_out.times * 1000, cal_avg.T * 1e6, lw=0.5, alpha=0.7)
axes[0].axvspan(DIPOLE_TMIN * 1000, DIPOLE_TMAX * 1000, color='tab:orange', alpha=0.3,
                label=f'dipole window [{DIPOLE_TMIN*1000:.0f}, {DIPOLE_TMAX*1000:.0f}] ms')
axes[0].set(xlabel='Time (ms)', ylabel='Amplitude (uV)',
            title=f'Calibration average after prime preprocessing '
                  f'(n = {int(cal_kept.sum())} trials, all {post_data.shape[1]} channels)')
axes[0].legend(fontsize=8)

for label, grp in trial_info_kept.groupby('block_label', sort=False):
    avg = post_data[grp.index.to_numpy()].mean(axis=0)
    axes[1].plot(epochs_out.times * 1000, np.sqrt((avg ** 2).mean(axis=0)) * 1e6,
                 lw=1.2, label=f'{label} (n={len(grp)})')
axes[1].axvspan(DIPOLE_TMIN * 1000, DIPOLE_TMAX * 1000, color='tab:orange', alpha=0.2)
axes[1].set(xlabel='Time (ms)', ylabel='GFP (uV)',
            title='Global field power of each block average')
axes[1].legend(fontsize=7, ncol=3)

fig.tight_layout()
fig.savefig(out_dir / f'sub-{subject}_ses-{session}_prime_preproc_summary.png', dpi=150)
plt.show()

In [ ]:
# --- residual-artifact scan inside the dipole window ------------------------------------
_in_w = (epochs_out.times >= DIPOLE_TMIN - 1e-9) & (epochs_out.times <= DIPOLE_TMAX + 1e-9)

# Group by condition where there is one, and by block otherwise. Only intervention-all has a
# 'condition' column (Section 2), so baseline + calibration + all four evaluation blocks have
# condition == '' -- grouping on condition alone would fold six blocks into a single
# meaningless '(none)' row and hide a residual artifact in any one of them. Falling back to
# block_label keeps every block separately visible, which is the whole point of the scan.
_cond = trial_info_kept['condition'].astype(str)
scan_key = _cond.where(_cond.astype(bool), trial_info_kept['block_label'])
scan_key.name = 'scan_group'

rows = []
for label, grp in trial_info_kept.groupby(scan_key, sort=False):
    avg = post_data[grp.index.to_numpy()].mean(axis=0)
    gfp = np.sqrt((avg ** 2).mean(axis=0)) * 1e6
    rows.append(dict(group=label,
                     source=('condition' if label in set(_cond[_cond.astype(bool)])
                             else 'block'),
                     n=len(grp),
                     gfp_in_window=gfp[_in_w].max(), gfp_overall=gfp.max(),
                     t_peak_ms=epochs_out.times[int(np.argmax(gfp))] * 1000))
scan_tbl = pd.DataFrame(rows).sort_values('gfp_in_window', ascending=False)

print(f"Peak GFP inside the dipole window [{DIPOLE_TMIN*1000:.0f}, {DIPOLE_TMAX*1000:.0f}] ms, "
      "by condition (blocks with no condition column are listed by block):")
print(scan_tbl.to_string(index=False, float_format=lambda v: f"{v:.2f}"))

_quiet = scan_tbl['gfp_in_window'].min()
_bad = scan_tbl[scan_tbl['gfp_in_window'] > 10 * _quiet]
if len(_bad):
    print(f"\n*** WARNING: {len(_bad)} group(s) exceed 10x the quietest "
          f"({_quiet:.2f} uV) inside the dipole window:")
    for _, r in _bad.iterrows():
        print(f"      {r['group']:<24} {r['gfp_in_window']:9.1f} uV "
              f"({r['gfp_in_window']/_quiet:.0f}x)  peak at {r['t_peak_ms']:.0f} ms "
              f"[{r['source']}]")
    print("    A TEP is ~1-3 uV; this magnitude is a TMS pulse, not brain activity.")
    print("    Most likely an unlogged extra pulse (e.g. a triplet's 2nd/3rd pulse) landing")
    print("    in the window.")
    print("    If the group is a CONDITION, add it to EXCLUDE_CONDITIONS in Section 1.")
    print("    If it is a BLOCK (baseline / evaluation-*), there is no condition to exclude:")
    print("    check that block's triggers in events.tsv against its trials_*.csv, since a")
    print("    whole block reading as artifact usually means a mis-mapped CSV or a bad cap.")
else:
    print(f"\nOK: no group exceeds 10x the quietest ({_quiet:.2f} uV) in the window.")

In [ ]:
# --- cross-check: calibrate()'s dipole_buffers vs preprocess_post on the same trials -----
# Works in either CALIBRATION_MODE: compare against whichever block actually calibrated.
_ref_label = CALIBRATION_STAGE if CALIBRATION_STAGE in cal_results else next(iter(cal_results))
dipole_buffers = cal_results[_ref_label]['dipole']
cal_kept = (trial_info_kept['block_label'] == _ref_label)

d0, d1 = DIPOLE_TMIN, DIPOLE_TMAX
sel = (np.isclose(epochs_out.times, d0) | np.isclose(epochs_out.times, d1)
       | ((epochs_out.times > d0) & (epochs_out.times < d1)))
post_dipole = post_data[cal_kept.to_numpy()][:, :, sel]

print(f"Cross-checking against the {_ref_label!r} calibration ({CALIBRATION_MODE} mode).\n")

# --- 1. structural alignment (channels and time samples) --------------------------------
# Both arrays come from the same Info, built by prime from forward.ch_names, so the channel
# axis is identical by construction. The time axis has to be checked: dipole_buffers is
# post_epochs.crop(dipole_tmin, dipole_tmax, include_tmax=True); post_dipole is the same
# window selected out of the wider saved epoch.
print(f"  dipole_buffers (from calibrate())      {dipole_buffers.shape}")
print(f"  post_dipole    (from preprocess_post)  {post_dipole.shape}")
print(f"  window selected: {int(sel.sum())} samples, "
      f"{epochs_out.times[sel][0]*1000:.0f}-{epochs_out.times[sel][-1]*1000:.0f} ms")

if post_dipole.shape[1:] != dipole_buffers.shape[1:]:
    print(f"  !! channel/time axes differ -- cannot compare numerically.")
else:
    # --- 2. why a positional comparison is NOT valid here ------------------------------
    # calibrate() drops trials internally at three points (ocular ICA, pre-stim MAD,
    # post-stim MAD) and returns only the survivors, exposing no index mapping.
    # preprocess_post() independently rejects per trial. The two survivor sets are
    # therefore DIFFERENT subsets of the same calibration block, and pairing them by
    # position (a[:n] vs b[:n]) lines trial i of one up against a different trial of the
    # other as soon as the first mismatch occurs. Measured on this dataset, a single
    # differently-dropped trial takes the positional correlation from 1.00 to ~0.95, and
    # four take it to ~0.06 -- which looks alarming but says nothing about preprocessing.
    n_cal_in = len(cal_results[_ref_label]['rows'])
    print(f"\n  {_ref_label}: {n_cal_in} trials in -> "
          f"{dipole_buffers.shape[0]} survived calibrate(), "
          f"{post_dipole.shape[0]} survived preprocess_post()")
    if dipole_buffers.shape[0] != post_dipole.shape[0]:
        print(f"  -> the two survivor sets differ in size, so they are NOT positionally "
              f"comparable.")

    def _corr(a, b):
        return float(np.corrcoef(a.ravel(), b.ravel())[0, 1])

    # --- 3. well-posed comparison: the trial-averaged topography ------------------------
    # Both are estimates of the same calibration evoked response over the dipole window,
    # and an average is robust to the two subsets differing by a handful of trials.
    r_avg = _corr(dipole_buffers.mean(axis=0), post_dipole.mean(axis=0))
    denom = np.abs(dipole_buffers.mean(axis=0)).max()
    rel = (np.abs(dipole_buffers.mean(axis=0) - post_dipole.mean(axis=0)).max() / denom
           if denom > 0 else np.inf)
    print(f"\n  trial-averaged topography : r = {r_avg:.4f}, "
          f"max|diff|/max|signal| = {rel:.4f}")

    # --- 4. do individual trials still correspond? --------------------------------------
    # For a sample of calibrate()'s trials, find the best-matching preprocess_post trial.
    # If the only problem is ordering, these match essentially perfectly.
    rng = np.random.default_rng(0)
    probe = rng.choice(dipole_buffers.shape[0],
                       size=min(20, dipole_buffers.shape[0]), replace=False)
    A = dipole_buffers.reshape(dipole_buffers.shape[0], -1)
    B = post_dipole.reshape(post_dipole.shape[0], -1)
    Az = (A - A.mean(1, keepdims=True)) / (A.std(1, keepdims=True) + 1e-30)
    Bz = (B - B.mean(1, keepdims=True)) / (B.std(1, keepdims=True) + 1e-30)
    best = (Az[probe] @ Bz.T / Az.shape[1]).max(axis=1)
    print(f"  median best-match per trial : r = {np.median(best):.4f} "
          f"(over {len(probe)} sampled trials)")

    if np.median(best) > 0.99 and r_avg > 0.95:
        print("\n  OK: trials correspond essentially exactly and the averages agree --")
        print("      the calibration state transfers correctly to preprocess_post.")
    elif np.median(best) > 0.99:
        print("\n  Trials correspond individually, but the averages differ more than")
        print("      expected -- check how many trials each stage dropped.")
    else:
        print("\n  WARNING: individual trials do NOT match well. This is a real difference")
        print("      between calibrate() and preprocess_post, not a bookkeeping artifact.")

print("\nDone. Point TEP_dipole_SpaTempFilt.ipynb's preproc_dir at this folder to continue.")